**Gaussian splatting local**

In [2]:
!pip install -r ../requirements.txt 

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [2]:
# Gaussian Splatting is a cutting-edge neural rendering technique used for high-quality, real-time 3D scene reconstruction and rendering. 
# Instead of representing a 3D scene as a mesh or voxel grid, Gaussian Splatting models the scene using a collection of 3D anisotropic Gaussians—each with its own position,
# orientation, color, and opacity.

# These Gaussians are "splatted" (projected) into the image space during rendering, allowing for continuous, 
# photorealistic views of the scene from any camera angle. This method bridges the gap between traditional point cloud rendering 
# and AI-powered NeRFs (Neural Radiance Fields), offering significantly faster rendering times while maintaining high visual fidelity.


In [3]:
# brush will be here ~/brush/target/release/brush_app 

In [40]:
# 1. Path declaration

import uuid
from pathlib import Path
import shutil

# User options from frontend
USE_DOWNSAMPLE = False
USE_REMBG = True

# Video file path (e.g. uploaded from frontend)
video_path = Path("/home/jovyan/mira3d/knife.MOV")  

# Setup UUID run directory
run_uuid = str(uuid.uuid4())
run_dir = Path("../data") / run_uuid
frames_dir = run_dir / "frames"
masked_dir = run_dir / "masked_frames"
colmap_dir = run_dir / "colmap"
brush_dir = run_dir / "brush"
run_dir.mkdir(parents=True)
frames_dir.mkdir()
if USE_REMBG:
    masked_dir.mkdir()
colmap_dir.mkdir()
brush_dir.mkdir()

# Copy video to run folder
input_video_path = run_dir / "input.mp4"
shutil.copy(video_path, input_video_path)

print(" Setup done:", run_uuid)


 Setup done: 8ab057dc-3c6e-4981-8cb1-970d4c172c75


In [41]:
# 2. Extract frames from video using ffmpeg with cuda acceleration

import subprocess

fps = "2"
subprocess.run([
    "ffmpeg","-hwaccel","cuda","-i", str(input_video_path),
    "-vf", f"fps={fps}",
    str(frames_dir / "%04d.jpg")
], check=True)

print("Frames extracted:", len(list(frames_dir.glob('*.jpg'))))


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Frames extracted: 103


In [42]:
# 3. Optional downsampling with ImageMagick

if USE_DOWNSAMPLE:
    downsample_dir = run_dir / "frames_2"
    downsample_dir.mkdir(exist_ok=True)

    # Resize all images to 50%
    !magick mogrify -path "{downsample_dir}" -resize 50%% "{frames_dir}/*.jpg"
    
    # Use downsampled frames for rest of pipeline
    frames_dir = downsample_dir

    print(" Images downsampled to 50%.")


In [44]:
# 4. Background removal using ONNX model (BirefNet)

if USE_REMBG:
    import onnxruntime as ort
    import numpy as np
    from PIL import Image
    from torchvision import transforms
    from tqdm import tqdm
    import gc

    # Load ONNX model and configure GPU execution
    onnx_model_path = "/home/jovyan/datafabric/birefnetgeneral-model/birefnet-general.onnx"
    session = ort.InferenceSession(onnx_model_path, providers=["CUDAExecutionProvider"])
    input_name = session.get_inputs()[0].name

    # Preprocessing pipeline
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -250, 250)))

    print("Removing backgrounds... please wait")

    for img_file in tqdm(frames_dir.glob("*.jpg"), desc="Processing frames"):
        # Load and preprocess
        img = Image.open(img_file).convert("RGB")
        w, h = img.size
        input_tensor = transform(img).unsqueeze(0).numpy().astype(np.float32)

        # Run inference
        pred = session.run(None, {input_name: input_tensor})[0].squeeze()
        mask = Image.fromarray((sigmoid(pred) * 255).astype(np.uint8)).resize((w, h))

        # Apply alpha mask
        mask_arr = np.array(mask) / 255.0
        result_img = img.copy()
        result_img.putalpha(Image.fromarray((mask_arr * 255).astype(np.uint8)))

        # Save masked image as PNG
        result_img.save(masked_dir / f"{img_file.stem}.png")

    # Use masked frames for the rest of the pipeline
    frames_dir = masked_dir
    print("Backgrounds removed using birefnet model.")

    # ---- FREE GPU MEMORY USED BY ONNX ----
    del session, onnx_model_path, input_tensor, pred, result_img, mask_arr, mask
    gc.collect()

    # Optional: show current memory usage
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        meminfo = pynvml.nvmlDeviceGetMemoryInfo(handle)
        print(f" ONNX GPU memory freed. Now using: {meminfo.used / 1024**2:.2f} MiB")
    except ImportError:
        print("ℹ️ pynvml not installed. Skipping GPU memory report.")



Removing backgrounds... please wait


Processing frames: 103it [01:16,  1.35it/s]

Backgrounds removed using birefnet model.
 ONNX GPU memory freed. Now using: 1263.56 MiB


In [45]:


# 5. Feature extraction: during this step COLMAP detects keypoints in each image using the
# SIFT (Scale-Invariant Feature Transform ) algorithm it will use gpu as out colmap with built with cuda
# --database_path: specifies where the features and matches will be stored(a sql lite db)
# -- iamgepath: the path to the folder containing our dataset images 


DB_PATH = colmap_dir / "database.db"

subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "PINHOLE",
    "--SiftExtraction.use_gpu", "1"
], check=True)

print(" COLMAP feature extraction complete.")
    


    

I0615 23:20:38.887602 159717 misc.cc:44] 
Feature extraction
I0615 23:20:38.888501 159730 sift.cc:721] Creating SIFT GPU feature extractor
I0615 23:20:39.243932 159731 feature_extraction.cc:259] Processed file [1/103]
I0615 23:20:39.244001 159731 feature_extraction.cc:262]   Name:            0001.png
I0615 23:20:39.244005 159731 feature_extraction.cc:271]   Dimensions:      2160 x 3840
I0615 23:20:39.244007 159731 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0615 23:20:39.244012 159731 feature_extraction.cc:277]   Focal Length:    4608.00px
I0615 23:20:39.244028 159731 feature_extraction.cc:281]   Features:        8623
I0615 23:20:41.055637 159731 feature_extraction.cc:259] Processed file [2/103]
I0615 23:20:41.055680 159731 feature_extraction.cc:262]   Name:            0002.png
I0615 23:20:41.055683 159731 feature_extraction.cc:271]   Dimensions:      2160 x 3840
I0615 23:20:41.055686 159731 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0615 23:20:41.05569

 COLMAP feature extraction complete.


In [46]:
# 6. COLMAP Matching The exhaustive matcher compares each image with every other image to 
# find matching points between them, which is crucial for the 3D reconstruction process

subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", str(DB_PATH),
    "--SiftMatching.use_gpu", "1"
], check=True)

print(" COLMAP matching complete.")


I0615 23:21:32.907967 160149 misc.cc:44] 
Feature matching
I0615 23:21:32.908418 160150 sift.cc:1428] Creating SIFT GPU feature matcher
I0615 23:21:33.017015 160149 pairing.cc:172] Generating exhaustive image pairs...
I0615 23:21:33.017028 160149 pairing.cc:205] Matching block [1/3, 1/3]
I0615 23:21:34.120719 160149 feature_matching.cc:47] in 1.104s
I0615 23:21:34.120739 160149 pairing.cc:205] Matching block [1/3, 2/3]
I0615 23:21:34.685966 160149 feature_matching.cc:47] in 0.565s
I0615 23:21:34.685986 160149 pairing.cc:205] Matching block [1/3, 3/3]
I0615 23:21:34.691181 160149 feature_matching.cc:47] in 0.005s
I0615 23:21:34.691191 160149 pairing.cc:205] Matching block [2/3, 1/3]
I0615 23:21:35.750226 160149 feature_matching.cc:47] in 1.059s
I0615 23:21:35.750247 160149 pairing.cc:205] Matching block [2/3, 2/3]
I0615 23:21:36.552871 160149 feature_matching.cc:47] in 0.803s
I0615 23:21:36.552891 160149 pairing.cc:205] Matching block [2/3, 3/3]
I0615 23:21:36.568972 160149 feature_matc

 COLMAP matching complete.


I0615 23:21:36.760875 160149 feature_matching.cc:47] in 0.092s
I0615 23:21:36.760895 160149 pairing.cc:205] Matching block [3/3, 3/3]
I0615 23:21:36.763491 160149 feature_matching.cc:47] in 0.003s
I0615 23:21:36.763504 160149 timer.cc:91] Elapsed time: 0.064 [minutes]


In [47]:
# 7. COLMAP Mapping (Sparse reconstruction) Structure from motion (SfM) is the process 
# of estimating the 3-D structure of a scene from a set of 2-D images. SfM is used in 
# many applications, such as 3-D scanning, augmented reality, and visual simultaneous localization and mapping (vSLAM)

SPARSE_PATH = colmap_dir / "sparse"
SPARSE_PATH.mkdir(exist_ok=True)

subprocess.run([
    "colmap", "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(frames_dir),
    "--output_path", str(SPARSE_PATH)
], check=True)

print(" COLMAP sparse reconstruction complete.")


I0615 23:21:43.646643 160246 incremental_pipeline.cc:253] Loading database
I0615 23:21:43.647660 160246 database_cache.cc:66] Loading rigs...
I0615 23:21:43.647683 160246 database_cache.cc:76]  1 in 0.000s
I0615 23:21:43.647687 160246 database_cache.cc:84] Loading cameras...
I0615 23:21:43.647696 160246 database_cache.cc:102]  1 in 0.000s
I0615 23:21:43.647701 160246 database_cache.cc:110] Loading frames...
I0615 23:21:43.647792 160246 database_cache.cc:127]  103 in 0.000s
I0615 23:21:43.647797 160246 database_cache.cc:135] Loading matches...
I0615 23:21:43.650416 160246 database_cache.cc:140]  1049 in 0.003s
I0615 23:21:43.650422 160246 database_cache.cc:156] Loading images...
I0615 23:21:43.657967 160246 database_cache.cc:238]  103 in 0.008s (connected 103)
I0615 23:21:43.657979 160246 database_cache.cc:249] Building correspondence graph...
I0615 23:21:43.669754 160246 database_cache.cc:276]  in 0.012s (ignored 0)
I0615 23:21:43.669816 160246 timer.cc:91] Elapsed time: 0.000 [minutes

 COLMAP sparse reconstruction complete.


I0615 23:22:28.917691 160246 incremental_pipeline.cc:571] Discarding reconstruction due to insufficient size
I0615 23:22:28.919654 160246 incremental_pipeline.cc:299] Finding good initial image pair
I0615 23:22:28.919680 160246 incremental_pipeline.cc:303] => No good initial image pair found.
I0615 23:22:28.919682 160246 incremental_pipeline.cc:538] Discarding reconstruction due to no initial pair
I0615 23:22:28.919737 160246 timer.cc:91] Elapsed time: 0.755 [minutes]


In [48]:
# Copy images into sparse/0 so Brush can find them
# 8. Copy images into sparse/0 so Brush can find them (PNG or JPG depending on input)
import shutil

image_output_dir = SPARSE_PATH / "0"
for img in frames_dir.glob("*.[jp][pn]g"): 
    shutil.copy(img, image_output_dir)


In [49]:
from IPython.display import display, Markdown, clear_output
import subprocess
from pathlib import Path

# 9. Run Brush
BRUSH_BIN = "/home/jovyan/brush/target/release/brush_app"
export_name = f"{run_uuid}.ply"
colmap_model_path = SPARSE_PATH / "0"

display_handle = display(Markdown(" **Processing... Please wait.**"), display_id=True)

try:
    subprocess.run([
        BRUSH_BIN,
        str(colmap_model_path),
        "--total-steps", "30000",
        "--export-path", str(brush_dir),
        "--export-name", export_name,
        "--export-every", "30000"
    ], check=True)

    display_handle.update(Markdown(f" **Brush reconstruction complete.** Output saved to: `{brush_dir / export_name}`"))

except subprocess.CalledProcessError as e:
    display_handle.update(Markdown(f" **Brush process failed.** Error: `{e}`"))


 **Brush reconstruction complete.** Output saved to: `../data/8ab057dc-3c6e-4981-8cb1-970d4c172c75/brush/8ab057dc-3c6e-4981-8cb1-970d4c172c75.ply`

error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.
error: XDG_RUNTIME_DIR not set in the environment.


In [50]:
# 10. Convert .ply to .splat using plyfile

from plyfile import PlyData
import numpy as np
from io import BytesIO

def process_ply_to_splat(ply_file_path):
    plydata = PlyData.read(ply_file_path)
    vert = plydata["vertex"]
    sorted_indices = np.argsort(
        -np.exp(vert["scale_0"] + vert["scale_1"] + vert["scale_2"])
        / (1 + np.exp(-vert["opacity"]))
    )
    buffer = BytesIO()
    for idx in sorted_indices:
        v = vert[idx]
        position = np.array([v["x"], v["y"], v["z"]], dtype=np.float32)
        scales = np.exp(
            np.array([v["scale_0"], v["scale_1"], v["scale_2"]], dtype=np.float32)
        )
        rot = np.array(
            [v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], dtype=np.float32
        )
        SH_C0 = 0.28209479177387814
        color = np.array(
            [
                0.5 + SH_C0 * v["f_dc_0"],
                0.5 + SH_C0 * v["f_dc_1"],
                0.5 + SH_C0 * v["f_dc_2"],
                1 / (1 + np.exp(-v["opacity"])),
            ]
        )
        buffer.write(position.tobytes())
        buffer.write(scales.tobytes())
        buffer.write((color * 255).clip(0, 255).astype(np.uint8).tobytes())
        buffer.write(
            ((rot / np.linalg.norm(rot)) * 128 + 128)
            .clip(0, 255)
            .astype(np.uint8)
            .tobytes()
        )
    return buffer.getvalue()

def save_splat_file(splat_data, output_path):
    with open(output_path, "wb") as f:
        f.write(splat_data)

ply_file_path = brush_dir / export_name
splat_file_path = ply_file_path.with_suffix(".splat")


print(f"Converting {ply_file_path} to SPLAT format...")
splat_data = process_ply_to_splat(ply_file_path)
save_splat_file(splat_data, splat_file_path)
print(f" SPLAT file saved to: {splat_file_path}")


Converting ../data/8ab057dc-3c6e-4981-8cb1-970d4c172c75/brush/8ab057dc-3c6e-4981-8cb1-970d4c172c75.ply to SPLAT format...
 SPLAT file saved to: ../data/8ab057dc-3c6e-4981-8cb1-970d4c172c75/brush/8ab057dc-3c6e-4981-8cb1-970d4c172c75.splat
